# Identity and PII Detection

This model predicts whether a document contains personally identifiable information (PII). Its output is binary: **PII detected** or **no PII detected**.

In [1]:
import pandas as pd

identity_df = pd.read_csv("identity_pii_binary_dataset.csv")

print("Dataset shape:", identity_df.shape)
print("\nPII label counts:")
print(identity_df["has_pii"].value_counts().sort_index())
print("\nMissing values:")
print(identity_df.isnull().sum())
print("\nSample rows:")
identity_df[["text", "has_pii", "variant"]].head()

Dataset shape: (100000, 4)

PII label counts:
has_pii
0    50000
1    50000
Name: count, dtype: int64

Missing values:
source_uid    0
variant       0
text          0
has_pii       0
dtype: int64

Sample rows:


,text,has_pii,variant
0,**ADOPTION CERTIFICATE**\nIssued by Guernsey A...,1,original
1,**ADOPTION CERTIFICATE**\nIssued by location A...,0,sanitized
2,**Account Closure Form**\n\n**Account Holder:*...,1,original
3,**Account Closure Form**\n\n**Account Holder:*...,0,sanitized
4,**Death Certificate**\n\n**Medical Record Numb...,1,original


## Split the Data

We will split by source document so each original document and its sanitized counterpart stay together. This prevents similar versions from appearing in both training and test data.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_indices, remaining_indices = next(
    group_split.split(
        identity_df,
        groups=identity_df["source_uid"]
    )
)

identity_train = identity_df.iloc[train_indices].copy()
identity_remaining = identity_df.iloc[remaining_indices].copy()

validation_test_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

validation_indices, test_indices = next(
    validation_test_split.split(
        identity_remaining,
        groups=identity_remaining["source_uid"]
    )
)

identity_validation = identity_remaining.iloc[validation_indices].copy()
identity_test = identity_remaining.iloc[test_indices].copy()

for name, data in [
    ("Training", identity_train),
    ("Validation", identity_validation),
    ("Test", identity_test),
]:
    print(f"{name} rows: {len(data)}")
    print(f"{name} unique source documents: {data['source_uid'].nunique()}")
    print(f"{name} label counts:")
    print(data["has_pii"].value_counts().sort_index())
    print()

Training rows: 70000
Training unique source documents: 35000
Training label counts:
has_pii
0    35000
1    35000
Name: count, dtype: int64

Validation rows: 15000
Validation unique source documents: 7500
Validation label counts:
has_pii
0    7500
1    7500
Name: count, dtype: int64

Test rows: 15000
Test unique source documents: 7500
Test label counts:
has_pii
0    7500
1    7500
Name: count, dtype: int64



## Convert Document Text to TF-IDF Features

We will represent the raw document text with single words and two-word phrases. The vectorizer will learn from training documents only.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

identity_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000
)

identity_train_features = identity_vectorizer.fit_transform(identity_train["text"])
identity_validation_features = identity_vectorizer.transform(identity_validation["text"])
identity_test_features = identity_vectorizer.transform(identity_test["text"])

print("Training feature shape:", identity_train_features.shape)
print("Validation feature shape:", identity_validation_features.shape)
print("Test feature shape:", identity_test_features.shape)

Training feature shape: (70000, 30000)
Validation feature shape: (15000, 30000)
Test feature shape: (15000, 30000)


## Train the PII Classifier

We will train a class-balanced Logistic Regression model to predict whether a document contains PII.

In [4]:
from sklearn.linear_model import LogisticRegression

identity_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

identity_model.fit(
    identity_train_features,
    identity_train["has_pii"]
)

print("PII classifier training complete.")

PII classifier training complete.


## Choose the PII Cutoff

We will choose the probability cutoff that gives the best F1 score on the validation documents. The test set remains reserved for final evaluation.

In [5]:
import numpy as np
from sklearn.metrics import precision_recall_curve

identity_probabilities = identity_model.predict_proba(identity_validation_features)[
    :, list(identity_model.classes_).index(1)
]

precision, recall, thresholds = precision_recall_curve(
    identity_validation["has_pii"],
    identity_probabilities
)

f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-10)
)

best_index = np.argmax(f1_scores)
identity_threshold = thresholds[best_index]

print("Chosen PII cutoff:", round(identity_threshold, 3))
print("Validation F1-score:", round(f1_scores[best_index], 3))

Chosen PII cutoff: 0.549
Validation F1-score: 0.99


## Evaluate on the Test Data

We will apply the cutoff chosen on the validation set to the untouched test documents and report precision, recall, F1 score, and the number of actual PII documents.

In [6]:
from sklearn.metrics import precision_recall_fscore_support

identity_test_probabilities = identity_model.predict_proba(identity_test_features)[
    :, list(identity_model.classes_).index(1)
]

identity_test_predictions = (
    identity_test_probabilities >= identity_threshold
).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(
    identity_test["has_pii"],
    identity_test_predictions,
    average="binary",
    zero_division=0
)

identity_results = pd.DataFrame(
    [{
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Actual PII documents": int(identity_test["has_pii"].sum())
    }],
    index=["PII"]
).round(3)

identity_results

,Precision,Recall,F1-score,Actual PII documents
PII,0.985,0.994,0.989,7500


## Save the PII Model

We will save the trained classifier, TF-IDF vectorizer, and validation-selected cutoff together so they can be reused.

In [7]:
import joblib

identity_bundle = {
    "model": identity_model,
    "vectorizer": identity_vectorizer,
    "threshold": identity_threshold,
    "label_mapping": {
        "no PII detected": 0,
        "PII detected": 1
    }
}

joblib.dump(identity_bundle, "identity_pii_model.joblib")
print("PII model saved.")

PII model saved.
